# Lab 8: Building and Improving a Document RAG System

In [ ]:
import os
from openai import OpenAI

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from IPython.display import display, Markdown

client = OpenAI()

## Business Context

I will be putting yourself in the shoes of a junior ML engineer (MLE) at Marlowe & Finch, where I have been asked to prototype a customer support assistant for the company's website.

#### 1. Company and Context

Marlowe & Finch is a small outdoor gear company based in Boulder, Colorado. The company was founded in 2017 by two former trail crew leads who got tired of gear that looked great in the store and fell apart on the trail. Today Marlowe & Finch designs lightweight, three-season backpacking gear for weekend backpackers and thru-hikers, and sells through its own website and a handful of independent outdoor retailers.

The customer base is mostly enthusiastic but not expert. Many customers are buying their first real piece of backcountry gear and have a lot of practical questions before they pull the trigger on a $400 tent.

#### 2. Business Challenge

Marlowe & Finch's customer support team currently answers most questions by hand, working from an internal knowledge base of product specs, warranty terms, and returns/shipping policies. Response times have been slipping as the volume of presale questions grows, and customers who do not get a quick answer often abandon their carts. The team has been asked to build a prototype assistant that can answer common product and policy questions on the website, using the same knowledge base the human agents reference today.

#### 3. Business Goal

The goal is to build a working RAG (Retrieval-Augmented Generation) prototype that can take a customer's question, retrieve the most relevant passages from the Marlowe & Finch knowledge base, and produce a grounded answer. If the prototype performs well, it would be deployed as a first-line assistant on the website. Customer support agents would handle anything the assistant cannot.

#### 4. My Role and Task

I have just joined Marlowe & Finch as a junior MLE on the Customer Experience team. The team has handed me the customer-facing knowledge base as a single text file and asked me to build and test a prototype RAG assistant. My job is to make the chunking, retrieval, and prompting decisions, then evaluate whether the prototype produces answers the customer support team would be comfortable putting in front of customers.

#### 5. Technical Focus in This Lab

This lab focuses on building and improving a document RAG pipeline:

* **Document Loading and Chunking** &mdash; Loading a source document and splitting it into focused chunks with `RecursiveCharacterTextSplitter`.
* **Vector Indexing and Retrieval** &mdash; Embedding chunks with OpenAI embeddings and storing them in a Chroma vector store for similarity search.
* **Prompt Engineering for RAG** &mdash; Writing an instruction prompt that grounds the model in retrieved context and handles questions the knowledge base cannot answer.
* **Chain Composition with LangChain** &mdash; Using the pipe operator to assemble a complete RAG chain.
* **Query Transformation** &mdash; Adding a query rewriting step in front of the retriever to improve retrieval on short, vague queries.

## Part 1. Load and Chunk the Knowledge Base

Marlowe & Finch has provided you with a single text file containing the customer-facing knowledge base. It is stored at `data/marlowe_knowledge_base.txt` and includes the company's "About" content, specs for three products, warranty terms, returns and shipping policies, and an FAQ.

Before you can search this knowledge base with embeddings, you need to load it and split it into chunks. You practiced both steps in the LangChain primer and the chunking strategies activity.

**Task**: Use `TextLoader` to load the file at `data/marlowe_knowledge_base.txt`. Save the result to a variable called `documents`. Then print how many documents you loaded and the total character length, so you can sanity-check that the file was read correctly.

*Tip*: `TextLoader(path).load()` returns a list of `Document` objects.

In [ ]:
loader = TextLoader("data/marlowe_knowledge_base.txt")
documents = loader.load()

print(f"Number of Documents: {len(documents)}")

total_chars = sum(len(doc.page_content) for doc in documents)
print(f"Total Character Length: {total_chars}")

Number of Documents: 1
Total Character Length: 12430


**Task**: Print the first 1,500 characters of the document so you can see what kind of content you are working with. You will use this view to inform your chunking choices in the next step.

*Tip*: You can slice a string with `text[:1500]`. The full document content lives in `documents[0].page_content`.

In [ ]:
print(documents[0].page_content[:1500])

MARLOWE & FINCH CUSTOMER SUPPORT KNOWLEDGE BASE
Last updated: Spring 2025

Welcome to the Marlowe & Finch customer support knowledge base. This document
is the internal source of truth our support team uses to answer customer
questions. It covers our current product lineup, warranty terms, returns and
shipping policies, and frequently asked questions.

ABOUT MARLOWE & FINCH

Marlowe & Finch is a small outdoor gear company based in Boulder, Colorado.
We design backcountry equipment for weekend backpackers and thru-hikers, with
a focus on lightweight three-season gear that holds up to real use. Our
products are sold through our website and a handful of independent outdoor
retailers. We do not currently sell through Amazon, REI, or other large
marketplaces.

We were founded in 2017 by two former trail crew leads who got tired of gear
that looked great in the store and fell apart on the trail. Every product in
our lineup is field-tested by our own staff before it ships.

PRODUCT CATALOG

-

Now that you have seen the content, take a moment to think about its structure. It mixes brand story, product specs, policy language, and FAQ entries. A chunk that mixes a product spec with an unrelated policy paragraph would be a noisy hit for either kind of query. A chunk that is too small might cut a policy sentence in half so that neither chunk contains a complete answer.

You will now split the document into chunks. The chunking strategies activity covered the trade-offs between small, medium, and large chunks, and the role of overlap.

**Task**: Create a `RecursiveCharacterTextSplitter` with `chunk_size` and `chunk_overlap` values of your choosing. Then call `.split_documents()` on `documents` to produce the chunks. Save the result to a variable called `chunks`.

After running your splitter, print the total number of chunks and the average chunk length so you can sanity-check your choice.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800 , chunk_overlap=100)
chunks = splitter.split_documents(documents)

num_chunks = len(chunks)
print(f"Number of Chunks: {num_chunks}")

avg_len = sum(len(chunk.page_content) for chunk in chunks) / num_chunks
print(f"Average Chunk Length: {avg_len}")

Number of Chunks: 20
Average Chunk Length: 619.55


**Task**: In the markdown cell below, answer the following:

1. What `chunk_size` and `chunk_overlap` did you choose?
2. Why did you pick those values for this particular knowledge base? Refer to something specific you saw in `documents[0].page_content`.
3. What is one risk of your choice (for example, what kind of customer question might it handle poorly)?

1. chunk_size=800, chunk_overlap=100
2. I chose a chunk size of 800 because it roughly fit the size of a full section that I saw in documents[0].page_contents[:1500]. I chose a chunk overlap of 100 characters because I believe that is enough to preserve context without excessive duplication across chunks. It is between 10%-20% of chunk size.
3. A risk of my choice is that some section in the knowledge base may be longer than 800 characters and the section would need to be split into multiple chunks. This might cause errors if the customer asks multiple questions about the same product. A chunk might be pulled that answers one of the questions, but not all of them.

## Part 2. Build the Vector Index

Now that you have chunks, you will embed them and store them in a vector database so the retriever can find the most relevant chunks for any customer query. You did this same pipeline in the Build a Vector Index activity.

**Task**: Initialize an `OpenAIEmbeddings` object using the `text-embedding-3-small` model. Save it to a variable called `embeddings`. Then build a Chroma vector store from your chunks using `Chroma.from_documents()`. Save the result to a variable called `vectorstore`.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

**Task**: Create a retriever from your vector store using `vectorstore.as_retriever()`. Configure it to use similarity search and to return the top 4 chunks. Save it to a variable called `retriever`.

*Tip*: `as_retriever()` accepts `search_type` and `search_kwargs` arguments. The `k` value goes inside `search_kwargs`.

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

Before you build the full RAG chain, let's inspect what the retriever returns for a real customer query, along with the similarity scores. The cell below uses `similarity_search_with_score()` so you can see how close each retrieved chunk actually is to the query.

In [ ]:
query = "What is your return policy?"
results = vectorstore.similarity_search_with_score(query, k=4)

for doc, score in results:
    print(f"Similarity Score: {score}\n")
    print(f"Chunk:\n{doc.page_content}")
    print("="*50)
    print("\n\n\n")

Similarity Score: 0.8796519637107849

Chunk:
To start a return:
1. Email returns@marloweandfinch.com with your order number.
2. We will email you a prepaid return label within one business day.
3. Pack the item in its original packaging and drop it off at any UPS
   location.
4. Refunds are issued within 5 business days of the item arriving back at
   our warehouse.

Sale items, gift cards, and customized gear (any product with a custom
embroidered name) are final sale and not eligible for return.

----------------------------------------------------------
Shipping
----------------------------------------------------------

We ship from our warehouse in Longmont, Colorado.

Standard shipping (within the continental US):
- Free on orders over $75
- $7.95 flat rate on orders under $75
- Delivery time: 3 to 5 business days




Similarity Score: 0.9872034788131714

Chunk:
You can return any unused item within 60 days of the purchase date for a
full refund to your original payment method.



**Task**: Look at the output above and think about how well the retriever is working. In the markdown cell below, answer the following:

1. Are the top retrieved chunks actually about the return policy, or are some of them off-topic?
2. What do the distance scores tell you? Is there a clear gap between the top result and the bottom result, or are they similar?
3. If you saw an off-topic chunk in the top 4, what do you think pulled it in? If all four chunks were on-topic, would your answer change if a customer asked something more specific, like "can I return a tent I used once"?

1. The top 2 chunks are related to the return policy. The 3rd chunk is about the warranty and the 4th chunk is about the company's sustainability. The 4th chunk is the most unrelated to the query, which makes sense that it was ranked below the other 3.
2. The distance scores tell me how closely related a chunk is to the query, where a lower distance means more closely related. There is a gap of about 0.4 between the top result and the bottom result.
3. I think the off-topic chunk (chunk number 4) was pulled in because the chunk is also related a policy. Asking a question about a policy in the original query with similar wording to the chunk's Q&A is likely what pulled it in.

## Part 3. Write the RAG Instruction Prompt

You have a retriever. Now you need an instruction prompt that tells the LLM what to do with the retrieved context. You wrote one of these in the Build Your First RAG System activity. In that activity, the prompt was for a generic question-answering assistant. Here you will write one tailored to Marlowe & Finch's customer support tone and to the specific risks of running a customer-facing assistant.

A good RAG prompt for this use case should:

1. Position the model as a customer support assistant for Marlowe & Finch
2. Tell the model to answer using only the retrieved context
3. Tell the model exactly what to say when the context does not contain the answer (for example, that it cannot find that information and the customer should contact customer support at `support@marloweandfinch.com`)
4. Set constraints on tone and length so the answers sound like the support team

**Task**: Create a variable called `rag_instruction` and assign it a prompt template string. The template must include the exact placeholders `{context}` and `{question}` so it can be used with `PromptTemplate.from_template()` later. Write the instruction text yourself, following the four requirements above.

*Tip*: Use triple quotation marks (`"""`) to write a multi-line string.

In [ ]:
rag_instruction = """
You are a friendly and helpful assistant for question-answering tasks for customers of Marlowe & Finch, a small outdoor gear company.
Use the following pices of retrieved context to answer the question.
If the context does not contain the answer to the question, say that you don't know and tell the customer to contact customer support at support@marloweandfinch.com.
Use 3 sentences maximum and keep the answer concise.

Context: {context}
Question: {question}
Answer:

"""

The cell below builds a `PromptTemplate` object from the string you wrote and prints the input variables it detected. You should see `['context', 'question']`.

In [ ]:
# Do not remove or edit this cell

prompt = PromptTemplate.from_template(rag_instruction)
print("PromptTemplate input variables:", prompt.input_variables)

PromptTemplate input variables: ['context', 'question']


## Part 4. Assemble the Baseline RAG Chain

Now you will assemble the complete RAG chain that you built in the Build Your First RAG System activity. The components are the same; only the corpus, the prompt, and the queries are different.

First, initialize the LLM.

In [ ]:
# Do not remove or edit this cell

llm = ChatOpenAI(model="gpt-4o")

**Task**: Write a function called `format_docs` that takes a list of `Document` objects and returns a single string containing their `page_content`, separated by two newlines (`\n\n`).

In [ ]:
def format_docs(docs):
    """
    Takes a list of Document objects and formats them into a single string.
    Each document's contents is separated by two newlines.
    """
    return "\n\n".join(doc.page_content for doc in docs)

**Task**: Assemble the baseline RAG chain. Save it to a variable called `rag_chain`. The chain should use:
- A dictionary mapping where `"context"` runs the retriever and pipes its output through `format_docs`, and `"question"` uses `RunnablePassthrough()`
- The `prompt` object created in Part 3
- The `llm` object initialized above
- A `StrOutputParser()` at the end

Use the pipe operator (`|`) to connect the components.

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Now let's test the chain on three customer queries. These are written to feel like what real shoppers actually type into a chat box. Pay attention to how the chain handles each one. They are deliberately different from each other:

- **Query 1** is short and underspecified, the kind of thing someone types when they have a quick question in their head.
- **Query 2** is a specific, scenario-based question about a real Marlowe & Finch policy.
- **Query 3** is a specific question that tests a particular edge case in Marlowe & Finch's policies.

In [ ]:
# Do not remove or edit this cell

baseline_queries = [
    "is the tent waterproof",
    "can i return a tent i used on a weekend trip",
    "do you ship to australia"
]

baseline_answers = {}

for q in baseline_queries:
    answer = rag_chain.invoke(q)
    baseline_answers[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("-" * 60)

Q: is the tent waterproof
A: The Trailhead 2 tent has a 2,000 mm hydrostatic head rating with fully taped seams, making it waterproof in steady rain and moderate wind. However, it is not suitable for sustained heavy rain, alpine storms, or snow loading. For those conditions, consider the Summit 4 four-season tent.

------------------------------------------------------------
Q: can i return a tent i used on a weekend trip
A: No, you cannot return a tent that has been used outdoors, even if it appears clean. If you believe the tent has a manufacturing defect, you can check if it falls under the Trail-Tested Warranty instead. For further assistance, contact customer support at support@marloweandfinch.com.

------------------------------------------------------------
Q: do you ship to australia
A: I don't know. Please contact customer support at support@marloweandfinch.com for assistance regarding shipping to Australia.

------------------------------------------------------------


**Task**: Review the three answers above. In the markdown cell below, answer the following:

1. How did the chain handle each of the three queries? Be specific: which ones produced clean answers, and which ones had problems? Were there any queries that the chain should have been able to answer but did not answer correctly? 
2. If a customer support manager saw these three answers, which one would they be most uncomfortable publishing on the website, and why?

1. The chain handled the queries fairly well. The first query was fully correct. The second query was answered correctly, but it provided the customer support email when it was not needed. However, the LLM was unable to retrieve an answer to the third query, even though there is an answer in knowledge base. The answer likely would've appeared if the user worded their question differently, such as "Where can you ship to?"
2. If a customer support manager saw these three answers, they would likely be uncomfortable publishing the third answer on the website since the LLM does not properly answer the question. The correct answer would be no, Marlowe and Finch does not ship to Australia.

## Part 5. Add Query Rewriting and Compare

Short, casual queries are a known weakness of vector retrieval. A query like "is the tent waterproof" is short and underspecified compared to the full-sentence documentation in the knowledge base.

In the query transformation activity, you saw that one way to address this is to rewrite the query into a longer, more complete version before sending it to the retriever. In this part, you will add a query rewriting step in front of the baseline chain and compare the results.

**Task**: Create a variable named `rewrite_prompt_template` that contains a prompt for rewriting a customer query. This prompt will be combined with the actual customer query using `.format()` in the next cell. Use the placeholder `{short_query}` where the customer's original query should appear.

Your prompt should:

1. Explain the task (rewrite a short customer query into a more complete version)
2. Make it clear that the rewritten query should preserve the customer's intent
3. Ask for exactly one rewritten query as output, not a list or numbered alternatives

*Note*: Because this template contains a curly-brace placeholder, define it as a regular string now and use `.format()` later, rather than as an f-string directly.

*Tip*: This is similar to the query rewriting work you did in the Improve RAG Retrieval Through Query Transformation activity, but there you asked for multiple alternatives. Here you want exactly one rewritten query so you can drop it straight into the existing retriever.

In [ ]:
# YOUR CODE HERE
rewrite_prompt_template = """
You are an expert at rewriting user queries to improve information retrieval from a customer-facing knowledge base which includes the company's "About" content, specs for three products, warranty terms, returns and shipping policies, and an FAQ.

Given the original query, generate a more complete version that expresses the same intent.
You should output exactly one rewritten query as output, not a list or numbered alternatives.

Original Query: {short_query}
Rewritten Query:

"""
# END OF YOUR CODE

The cell below uses your `rewrite_prompt_template` to rewrite each of the three queries from Part 4. To keep things fast and consistent, we rewrite each query once and save the results in a dictionary called `rewritten_queries`. That way, when we compare baseline answers to rewritten-query answers, we are using the same rewrites throughout.

In [ ]:
# Do not remove or edit this cell

rewritten_queries = {}

for q in baseline_queries:
    filled_prompt = rewrite_prompt_template.format(short_query=q)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": filled_prompt}]
    )
    rewritten_queries[q] = response.choices[0].message.content.strip()
    print(f"Original:  {q}")
    print(f"Rewritten: {rewritten_queries[q]}\n")

Original:  is the tent waterproof
Rewritten: Can you provide information on whether the tent is waterproof, including any specific specifications or features related to its water resistance?

Original:  can i return a tent i used on a weekend trip
Rewritten: Am I eligible to return a tent that I used during a weekend trip, and what is the process if it meets the return policy criteria?

Original:  do you ship to australia
Rewritten: Do you offer shipping to Australia and what are the details of the shipping policy to this location?



Now let's run the full pipeline (the rewritten query through the existing RAG chain) and compare the answers side by side with the baseline.

In [ ]:
# Do not remove or edit this cell

for q in baseline_queries:
    rewritten = rewritten_queries[q]
    rewritten_answer = rag_chain.invoke(rewritten)

    print(f"Original query: {q}")
    print(f"Rewritten query: {rewritten}\n")
    display(Markdown(f"**Baseline answer:** {baseline_answers[q]}"))
    display(Markdown(f"**Rewritten-query answer:** {rewritten_answer}"))
    print("=" * 60)

Original query: is the tent waterproof
Rewritten query: Can you provide information on whether the tent is waterproof, including any specific specifications or features related to its water resistance?



**Baseline answer:** The Trailhead 2 tent has a 2,000 mm hydrostatic head rating with fully taped seams, making it waterproof in steady rain and moderate wind. However, it is not suitable for sustained heavy rain, alpine storms, or snow loading. For those conditions, consider the Summit 4 four-season tent.

**Rewritten-query answer:** The Trailhead 2 tent has a 2,000 mm hydrostatic head rating on both the fly and the floor and features fully taped seams, which makes it suitable for steady rain and moderate wind. However, it is not rated for sustained heavy rain, alpine storms, or snow loading. For such conditions, the Summit 4 four-season tent is recommended.

Original query: can i return a tent i used on a weekend trip
Rewritten query: Am I eligible to return a tent that I used during a weekend trip, and what is the process if it meets the return policy criteria?



**Baseline answer:** No, you cannot return a tent that has been used outdoors, even if it appears clean. If you believe the tent has a manufacturing defect, you can check if it falls under the Trail-Tested Warranty instead. For further assistance, contact customer support at support@marloweandfinch.com.

**Rewritten-query answer:** You are not eligible to return a tent that has been used outdoors, even if it appears clean. If your tent meets the criteria for unused items, you can return it within 60 days of purchase. For further assistance, please contact customer support at support@marloweandfinch.com.

Original query: do you ship to australia
Rewritten query: Do you offer shipping to Australia and what are the details of the shipping policy to this location?



**Baseline answer:** I don't know. Please contact customer support at support@marloweandfinch.com for assistance regarding shipping to Australia.

**Rewritten-query answer:** I don't know if we offer shipping to Australia. Please contact customer support at support@marloweandfinch.com for more information.

**Task**: Review the side-by-side comparison above. In the markdown cell below, answer the following:

1. On which of the three queries did query rewriting visibly change the answer? Did it improve the answer, make it worse, or leave it about the same?
2. Look at the rewritten queries themselves. Are they faithful to the original intent, or did the rewriting step change what the customer was asking about?
3. Query rewriting adds an extra LLM call before every retrieval, which costs both money and a bit of latency. Based on what you saw, is the improvement worth the added cost for this use case? When would you turn it on, and when would you leave it off?

1. All three queries remained about the same. I think this is because the first two queries already had fairly solid responses. The third query was still unable to be answered. I think we may have had better luck with that third query if we offered multiple question rewrites instead of just one. Also, I think the LLM is having trouble answering the question because the knowledge base lists the names of the areas it does ship to, but it does not name the areas it does not ship to.
2. The rewritten queries are faithful to the original intent.
3. Based on what I saw, adding one rewrite was not worth the added cost. However, it may have been more useful if we used multiple rewrites instead of just one. I would turn the rewrites on when the LLM would provide an unsure answer with the original query. I would leave it on otherwise.

## Part 6. Analysis

You have now built a complete RAG assistant for Marlowe & Finch and tested a query rewriting improvement on top of it. In this section, reflect on the system as a whole.

Answer the following questions in the markdown cell below:

1. **Explaining the system to the team**: Marlowe & Finch's customer support manager is not technical. They want to know, in plain language, why the assistant sometimes gives a great answer and sometimes gives a vague one. Using what you observed in this lab, write a short explanation (3-5 sentences) you could give the manager. Avoid complicated technical jargon. Do not assume they know what an embedding is.

2. **Biggest deployment risk**: Marlowe & Finch is considering putting this assistant live on the website as a first-line responder to customer questions. Based on what you observed in this lab, what is the single biggest risk of deploying this prototype as-is? Name one specific type of customer question that would likely cause the assistant to behave badly, and explain what you would recommend to the team before going live.

1. The assistant sometimes gives a vauge answer because every time we use the assistant, its response is simply a prediction based on the information we provide it. The assistant does not know what is fact and what is not, so sometimes it will miss important details and focus on the less important ones. The phrasing of a question can also impact performance, as certain words will align better with information that is in the knowledge base. Even slightly different wordings can change the information that the assistant looks for in the knoweldge base.
2. The single biggest risk of deploying this prototype as-is is receiving many support emails for simple questions. An instance of this is if the customer asks "Do you ship to Australia?", the model will respond that it is unsure and instruct the customer to email the support team. To combat the model's inability to answer questions like this, you might want to expand your company's knowledge base to include emails sent to support with the responses so that ideally, the support team will not need to answer the same question twice.

## Part 7. Reflection: AI Usage

1. Did you use AI tools for this lab? If yes, which ones and at what points in your work? If no, briefly explain your reasoning.
2. If you used AI, describe one specific prompt that was useful and explain why it worked. If you did not use AI, walk through one part of the lab where you had to figure something out on your own and explain how you got there.
3. How did you verify that your work was correct? What would you look for to catch a mistake, whether it came from AI or from your own reasoning?
4. What is one thing you would do differently next time, either in how you approached the lab or in how you used (or did not use) AI?


Record your findings in the cell below.

1. I did use AI tools for this lab. I used Claude whenever I was unsure about something or didn't understand why I got a certain output.
2. One prompt that was useful was asking why I was not seeing improvement in the query about shipping to Australia after rewriting the prompt. Using Claude here was useful because it helped me understand some factors that likely contribute to this, such as the temperature being too low.
3. I verified that my work was correct by inspecting the outputs. If something appeared off, I would try to figure out the mistake myself. If I could not figure it out on my own, I would ask AI for help.
4. Next time, I would have multiple rewrites of queries instead of just one so that there is a better chance of getting the answer I was looking for.